<a href="https://colab.research.google.com/github/EdjeElectronics/Train-and-Deploy-YOLO-Models/blob/main/Train_YOLO_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍌 YOLOv11 Training - Deteksi Kematangan Buah Sawo

Training model YOLOv11 untuk mendeteksi 2 kelas kematangan buah sawo:
- **Belum Matang** (termasuk setengah matang)
- **Matang**

Notebook ini akan:
1. Install dependencies
2. Setup dataset dari Google Drive
3. Train model YOLOv11
4. Evaluasi dan visualisasi hasil
5. Export ke format ONNX untuk deployment

## 1️⃣ Install Dependencies

In [ ]:
!pip install -q ultralytics opencv-python torch torchvision
print("✅ Dependencies installed!")

## 2️⃣ Mount Google Drive (Untuk Dataset)

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')
print("✅ Google Drive mounted!")

# Setup paths
base_path = Path('/content/drive/MyDrive/Sawo_Training')
dataset_path = base_path / 'dataset'
output_path = base_path / 'outputs'

output_path.mkdir(parents=True, exist_ok=True)
print(f"📁 Dataset path: {dataset_path}")
print(f"📁 Output path: {output_path}")

## 3️⃣ Validasi Dataset

In [ ]:
import os
from pathlib import Path

# Check dataset structure
print("📋 Struktur Dataset YOLO:")
print("""
dataset/
  ├── train/
  │   ├── images/  (gambar training)
  │   └── labels/  (label .txt format YOLO)
  ├── valid/
  │   ├── images/  (gambar validation)
  │   └── labels/  (label .txt format YOLO)
  └── assets/      (opsional, data dikelompokkan per kelas)
""")

# Count files
train_images = list((dataset_path / 'train' / 'images').glob('*'))
val_images = list((dataset_path / 'valid' / 'images').glob('*'))
train_labels = list((dataset_path / 'train' / 'labels').glob('*.txt'))
val_labels = list((dataset_path / 'valid' / 'labels').glob('*.txt'))

print(f"\n✓ Training images: {len(train_images)}")
print(f"✓ Validation images: {len(val_images)}")
print(f"✓ Training labels: {len(train_labels)}")
print(f"✓ Validation labels: {len(val_labels)}")

def _find_invalid_class_id(label_paths, sample_limit=500):
    for p in label_paths[:sample_limit]:
        try:
            lines = p.read_text(encoding='utf-8', errors='ignore').splitlines()
        except Exception:
            continue
        for line in lines:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if not parts:
                continue
            try:
                cid = int(float(parts[0]))
            except Exception:
                continue
            if cid not in (0, 1):
                return p.name, cid
    return None

invalid = _find_invalid_class_id(train_labels + val_labels)
if invalid:
    fname, cid = invalid
    raise ValueError(
        f"Label tidak sesuai 2-kelas. File {fname} punya class id {cid}. "
        "Gunakan mapping: 0=belum_matang, 1=matang."
    )

if len(train_images) == 0:
    print("\n⚠️  PERHATIAN: Tidak ada data training!")
    print("   Upload dataset ke Google Drive terlebih dahulu")
    print("   Path: /MyDrive/Sawo_Training/dataset/")
else:
    print("\n✅ Dataset siap untuk training!")

## 4️⃣ Buat Konfigurasi data.yaml

In [ ]:
import yaml

# Create data.yaml
yaml_content = {
    'path': str(dataset_path),
    'train': 'train/images',
    'val': 'valid/images',
    'nc': 2,
    'names': {
        0: 'belum_matang',
        1: 'matang'
    }
}

yaml_path = dataset_path / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f"✓ Konfigurasi tersimpan: {yaml_path}")
print("\nIsi data.yaml:")
print(yaml_content)

## 5️⃣ Train Model YOLOv11

In [ ]:
from ultralytics import YOLO
import torch

# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

# Load model
print("\n📥 Loading YOLOv11n (nano model)...")
model = YOLO('yolov11n.pt')

# Train
print("\n🚀 Memulai training...")
results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,           # Early stopping
    save=True,
    device=0,              # GPU 0
    project=str(output_path),
    name='sawo_detection',
    exist_ok=True,
    plots=True,            # Generate plots
    verbose=True
)

print("\n✅ Training selesai!")

## 6️⃣ Visualisasi Hasil Training

In [ ]:
import matplotlib.pyplot as plt
import json
import numpy as np
from pathlib import Path

# Find latest run
run_dir = output_path / 'sawo_detection'
if not run_dir.exists():
    candidates = sorted(output_path.glob('sawo_detection*'))
    run_dir = candidates[-1] if candidates else None

if run_dir:
    results_json = run_dir / 'results.json'
    
    if results_json.exists():
        with open(results_json) as f:
            data = json.load(f)
        
        # Plot metrics
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('🎯 Training Metrics - YOLOv11 Sawo Detection', fontsize=16, fontweight='bold')
        
        epochs = list(range(len(data['train/loss'])))
        
        # 1. Loss
        axes[0, 0].plot(epochs, data['train/loss'], label='Train Loss', marker='o')
        axes[0, 0].plot(epochs, data['val/loss'], label='Val Loss', marker='s')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].set_title('📉 Loss (Training vs Validation)')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. mAP50
        axes[0, 1].plot(epochs, data['metrics/mAP50(B)'], label='mAP50', color='green', marker='o')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('mAP50 (%)')
        axes[0, 1].set_title('📊 Mean Average Precision @ IoU=0.5')
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. mAP50-95
        axes[1, 0].plot(epochs, data['metrics/mAP50-95(B)'], label='mAP50-95', color='purple', marker='o')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('mAP50-95 (%)')
        axes[1, 0].set_title('📈 Mean Average Precision @ IoU=0.5-0.95')
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Overfitting Check
        train_loss = np.array(data['train/loss'])
        val_loss = np.array(data['val/loss'])
        gap = val_loss - train_loss
        axes[1, 1].fill_between(epochs, 0, gap, where=(gap > 0), alpha=0.5, color='red', label='Overfitting')
        axes[1, 1].fill_between(epochs, 0, gap, where=(gap <= 0), alpha=0.5, color='green', label='Good Fit')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Validation - Training Loss')
        axes[1, 1].set_title('🔍 Overfitting Detection')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        axes[1, 1].axhline(y=0, color='black', linestyle='--', linewidth=1)
        
        plt.tight_layout()
        plt.savefig(output_path / 'training_metrics.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("✅ Grafik tersimpan: training_metrics.png")
    else:
        print("❌ results.json tidak ditemukan")
else:
    print("❌ Tidak ada training run")

## 7️⃣ Analisis Overfitting

In [ ]:
import numpy as np

if results_json.exists():
    with open(results_json) as f:
        data = json.load(f)
    
    train_loss = np.array(data['train/loss'])
    val_loss = np.array(data['val/loss'])
    
    # Hitung statistik
    avg_train_loss = np.mean(train_loss[-10:])  # Last 10 epochs
    avg_val_loss = np.mean(val_loss[-10:])
    gap = avg_val_loss - avg_train_loss
    
    mAP50 = max(data['metrics/mAP50(B)']) * 100
    mAP50_95 = max(data['metrics/mAP50-95(B)']) * 100
    
    print("="*60)
    print("📊 ANALISIS OVERFITTING & PERFORMA MODEL")
    print("="*60)
    
    print(f"\n📈 Metrik Performa:")
    print(f"   • mAP50 (Best):     {mAP50:.2f}%")
    print(f"   • mAP50-95 (Best):  {mAP50_95:.2f}%")
    
    print(f"\n📉 Loss Analysis (Last 10 Epochs):")
    print(f"   • Training Loss:    {avg_train_loss:.4f}")
    print(f"   • Validation Loss:  {avg_val_loss:.4f}")
    print(f"   • Gap (Val - Train): {gap:.4f}")
    
    # Overfitting check
    print(f"\n🔍 Status Overfitting:")
    if gap < 0.05:
        print(f"   ✅ BAIK - Model tidak overfitting (gap: {gap:.4f})")
        print(f"      Training & validation loss seimbang")
    elif gap < 0.1:
        print(f"   ⚠️  MODERATE - Sedikit overfitting (gap: {gap:.4f})")
        print(f"      Pertimbangkan lebih banyak data atau augmentasi")
    else:
        print(f"   ❌ TINGGI - Overfitting detected (gap: {gap:.4f})")
        print(f"      Perlukan lebih banyak data training")
    
    print(f"\n💡 Rekomendasi:")
    if mAP50 > 0.85:
        print(f"   ✅ Model performa SANGAT BAIK! Ready untuk production")
    elif mAP50 > 0.7:
        print(f"   ✅ Model performa BAIK, bisa digunakan")
    else:
        print(f"   ⚠️  Model performa cukup, pertimbangkan tambah epochs atau data")
    
    print("\n" + "="*60)

## 8️⃣ Evaluasi Model pada Validation Set

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Load best model
run_dir = output_path / 'sawo_detection'
if not run_dir.exists():
    candidates = sorted(output_path.glob('sawo_detection*'))
    run_dir = candidates[-1] if candidates else None

if run_dir:
    best_model_path = run_dir / 'weights' / 'best.pt'
    
    print(f"🔍 Loading model: {best_model_path}")
    model = YOLO(str(best_model_path))
    
    # Validate
    print("\n📊 Evaluating on validation set...")
    metrics = model.val(data=str(yaml_path))
    
    print("\n✅ Evaluasi selesai!")
    print(f"   mAP50: {metrics.box.map50:.3f}")
    print(f"   mAP50-95: {metrics.box.map:.3f}")

## 9️⃣ Export ke Format ONNX (untuk Deploy)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

# Find best model
run_dir = output_path / 'sawo_detection'
if not run_dir.exists():
    candidates = sorted(output_path.glob('sawo_detection*'))
    run_dir = candidates[-1] if candidates else None

if run_dir:
    best_model_path = run_dir / 'weights' / 'best.pt'
    
    print(f"🔄 Exporting to ONNX format...")
    model = YOLO(str(best_model_path))
    
    export_path = model.export(
        format='onnx',
        imgsz=640,
        half=False,
        opset=14,
        device=0
    )
    
    print(f"✅ Model ONNX exported: {export_path}")
    
    # Copy to output folder
    output_onnx = output_path / 'best.onnx'
    shutil.copy(export_path, output_onnx)
    
    print(f"\n📥 Model tersimpan di:")
    print(f"   {output_onnx}")
    print(f"\n💡 Download file ini dan copy ke: /public/models/best.onnx")
    print(f"   di project React Anda")

## 🔟 Test Inference pada Sample Image

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from PIL import Image

# Find best model
run_dir = output_path / 'sawo_detection'
if not run_dir.exists():
    candidates = sorted(output_path.glob('sawo_detection*'))
    run_dir = candidates[-1] if candidates else None

if run_dir:
    best_model_path = run_dir / 'weights' / 'best.pt'
    model = YOLO(str(best_model_path))
    
    # Get sample image dari validation set
    val_images = list((dataset_path / 'valid' / 'images').glob('*'))
    
    if val_images:
        sample_image = str(val_images[0])
        print(f"🖼️  Testing inference on: {val_images[0].name}")
        
        # Inference
        results = model.predict(sample_image, conf=0.5)
        
        # Plot
        result = results[0]
        im_array = result.plot()
        
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB))
        plt.title('🎯 Inference Result')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(output_path / 'inference_test.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("\n✅ Inference test selesai!")
        print(f"   Detections: {len(result.boxes)}")
        for box in result.boxes:
            class_name = result.names[int(box.cls)]
            conf = float(box.conf)
            print(f"   • {class_name}: {conf:.2%}")
    else:
        print("❌ Tidak ada validation image")

## 📋 Summary & Next Steps

✅ **Training selesai!**

### Hasil tersimpan di:
- 📁 Model: `/outputs/sawo_detection/weights/best.pt`
- 📁 ONNX: `/outputs/best.onnx`
- 📊 Grafik: `/outputs/training_metrics.png`
- 📊 Test: `/outputs/inference_test.png`

### Langkah Selanjutnya:
1. ⬇️ **Download file** `best.onnx` dari output folder
2. 📂 **Copy ke project React**: `/public/models/best.onnx`
3. 🔄 **Reload aplikasi** di browser
4. 🎯 **Test deteksi** di halaman `/detect`

### Struktur Model:
- **Classes**: belum_matang, matang
- **Input**: 640x640 RGB image
- **Output**: Bounding boxes dengan confidence scores
- **Framework**: YOLOv11 (ONNX format)

Selamat! 🎉